## Permissions
Minimum permissions to run this are:
* Monitoring Metrics Publisher on DCR to write
* Monitoring Reader on DCR
* Monitoring Reader on DCE

# Setup

In [0]:
%pip install {dbutils.widgets.get("lib_path") + "/.internal/*.whl"}
%restart_python

In [ ]:
from databricks.sdk.runtime import dbutils, spark

In [0]:
import json

from azure.identity import ClientSecretCredential

from pyspark.sql import functions as F

from sentinel_helpers.log_analytics import get_dce, get_dcr
from sentinel_helpers.spark import AzureMonitorDataSource

spark.dataSource.register(AzureMonitorDataSource)

# Variables

In [0]:
dbutils.widgets.text("log_analytics_table_name", "")
dbutils.widgets.text("system_table_name", "")
dbutils.widgets.text("checkpoint_volume_location", "")
dbutils.widgets.text("starting_datetime", "")
dbutils.widgets.text("include_workspace_ids", "")
dbutils.widgets.text("exclude_workspace_ids", "")
dbutils.widgets.text("processing_time", "1 minute")

dbutils.widgets.text("tenant_id", "")
dbutils.widgets.text("subscription_id", "")
dbutils.widgets.text("sp_client_id", "")
dbutils.widgets.text("sp_client_secret_scope", "")
dbutils.widgets.text("sp_client_secret_key", "")
dbutils.widgets.text("resource_group_name", "")

In [0]:
log_analytics_table_name = dbutils.widgets.get("log_analytics_table_name")
system_table_name = dbutils.widgets.get("system_table_name")
checkpoint_volume_location = dbutils.widgets.get("checkpoint_volume_location")
starting_datetime = dbutils.widgets.get("starting_datetime")
include_workspace_ids = json.loads(dbutils.widgets.get("include_workspace_ids"))
exclude_workspace_ids = json.loads(dbutils.widgets.get("exclude_workspace_ids"))
processing_time = dbutils.widgets.get("processing_time")

tenant_id = dbutils.widgets.get("tenant_id")
subscription_id = dbutils.widgets.get("subscription_id")
sp_client_id = dbutils.widgets.get("sp_client_id")
sp_client_secret_scope = dbutils.widgets.get("sp_client_secret_scope")
sp_client_secret_key = dbutils.widgets.get("sp_client_secret_key")
resource_group_name = dbutils.widgets.get("resource_group_name")

sp_secret = dbutils.secrets.get(sp_client_secret_scope, sp_client_secret_key)

# Initialization

In [0]:
data_collection_endpoint_name = f"{log_analytics_table_name}-dce"
data_collection_rule_name = f"{log_analytics_table_name}-dcr"
raw_stream_declaration_name = f"Custom-{log_analytics_table_name}RawData"

credentials = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=sp_client_id,
    client_secret=sp_secret
)

dcr_id = get_dcr(credentials, subscription_id, resource_group_name, data_collection_rule_name).immutable_id
dce_url = get_dce(credentials, subscription_id, resource_group_name, data_collection_endpoint_name).logs_ingestion.endpoint

sink_options = {
    "dce_url": dce_url,
    "dcr_id": dcr_id,
    "dcs": raw_stream_declaration_name,
    "tenant_id": tenant_id,
    "client_id": sp_client_id,
    "client_secret": sp_secret,
    "body_col": "row_json",
}

## Core

In [0]:
input_df = (
    spark.readStream
    .option("skipChangeCommits", "true")
    .table(system_table_name)
    .filter(F.col("event_time") >= starting_datetime)
)

if include_workspace_ids:
    input_df = input_df.filter(F.col("workspace_id").isin(include_workspace_ids))
if exclude_workspace_ids:
    input_df = input_df.filter(~F.col("workspace_id").isin(exclude_workspace_ids))

streaming_query = (
    input_df.withColumn("TimeGenerated", F.col("event_time"))
    .select(F.to_json(F.struct("*")).alias("row_json"))
    .writeStream.format("azure-monitor")
    .option("checkpointLocation", checkpoint_volume_location)
    .options(**sink_options)
    .outputMode("append")
    .queryName(f"{system_table_name} streaming process")
    
)

In [0]:
if processing_time:
    print(f"⏳ Starting streaming query with processing time: {processing_time}")
    streaming_query.trigger(processingTime=processing_time).start()
else:
    print(f"⏳ Starting streaming query with availableNow trigger")
    q = streaming_query.trigger(availableNow=True).start()
    q.awaitTermination()
    print("✅ Processing completed !")